# NETRA-4D verified Colab T4 pipeline

This notebook runs real Ultralytics pose and object models against the five user-supplied Google Drive videos. It has no simulated detections, fixed fallback boxes, seeded events, fabricated benchmark scores, or synthetic 3D joints. Dataset filenames are source labels, not predictions.

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'scripts' / 'run_verified_inference.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'run_verified_inference.py').exists(), 'Open this notebook from the NETRA-4D repository.'
os.chdir(ROOT)
print('Repository:', ROOT)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'A CUDA runtime is required. In Colab select Runtime > Change runtime type > T4 GPU.'
print({
    'torch': torch.__version__,
    'cuda_build': torch.version.cuda,
    'device': torch.cuda.get_device_name(0),
    'vram_bytes': torch.cuda.get_device_properties(0).total_memory,
})

In [ ]:
from pathlib import Path
import shutil
from ultralytics import YOLO

weight_dir = ROOT / 'models' / 'weights'
weight_dir.mkdir(parents=True, exist_ok=True)
for name in ('yolo26n-pose.pt', 'yolo11n.pt'):
    target = weight_dir / name
    if not target.exists():
        loaded = YOLO(name)
        source = Path(loaded.ckpt_path).resolve()
        shutil.copy2(source, target)
    print(name, target.stat().st_size, 'bytes')

In [ ]:
import subprocess

completed = subprocess.run([
    sys.executable, 'scripts/run_verified_inference.py',
    '--device', 'cuda:0', '--require-cuda',
    '--max-frames', '20',
], check=True, text=True, capture_output=True)
print(completed.stdout)

In [ ]:
import json
from IPython.display import display, Image

result_path = max((ROOT / 'outputs' / 'verified').glob('*/run.json'), key=lambda p: p.stat().st_mtime)
result = json.loads(result_path.read_text(encoding='utf-8'))
assert result['runtime']['actual_device'] == 'cuda:0'
display(result['summary'])
for asset in result['assets']:
    print(asset['source']['asset_id'], asset['source']['filename'])
    for frame in asset['sampled_frames'][:1]:
        display(Image(filename=str(result_path.parent / frame['preview'])))